In [1]:
import sys
from pathlib import Path

here = Path.cwd()
repo_root = here.parents[1] if here.name == "mrna_features" else here
sys.path.insert(0, str(repo_root))

from utils.merge_historic_data import load_merged_dataset
from utils.pipeline import SiRNADataPipeline
from utils.fm_utils import add_slice_columns
from utils.mrna_features.gc_content_mfe import add_binding_features, add_utr_gc, spearman_vs_inhibition

data_dir = Path("/Users/alinalapkovskaya/Documents/TUM/ML4RG/siRNA_project/data")
cm = data_dir/"CMsiRNA_data_update.tsv"
hist = data_dir/"Historic_Takayuki_hueskan_ichihara.csv"

raw = load_merged_dataset(cm, hist)
pipeline = SiRNADataPipeline(target_len=25)
enriched = pipeline.enrich_dataset_with_encodings(raw, strict_cleaning=False, add_mrna=True)
enriched = add_slice_columns(enriched)

loaded 3515 historic rows
merged 43153 CMsiRNA and 3515 historic rows into 46668
Running qc and data cleaning
dropped 5756 rows (in-vivo 4233, mM 565, conc>200 797, cell 115, inhibition 102)
dropped 1749 rows with NaN concentration
dropped 2198 rows with a missing or >25 nt strand
imputed 6091 missing time rows (median 24.0h at or below 10 nM, 48.0h above)
dropped 5 columns: ['Modification_locations_Sense_strand', 'Modification_locations_Antisense_strand', 'Modifications_sense_strand', 'Modifications_AntiSense_strand_3_5', 'position_Sense_strand']
Mapping mRNA structural profiles
Error reading reference dataset: [Errno 2] No such file or directory: '/home/larsena8/software/fennec/src/fennec/support_files/train_data_v1.1.0_N=27742.csv'
Loaded 47 gene sequences from local cache
Loaded 0 gene sequences from reference CSV
Building gene -> mRNA mapping for 54 genes...
[1/54] Processing CTNNB1...
  Found in local cache (3074 bp)
[2/54] Processing INHBE...
  Found in local cache (2460 bp)
[3/

In [2]:
enriched = add_binding_features(enriched, with_mfe=False)
gc_cols = ["binding_gc", "binding_gc_start", "binding_gc_mid", "binding_gc_end"]
spearman_vs_inhibition(enriched, gc_cols)

,feature,spearman_overall,spearman_by_gene,n
0,binding_gc,-0.089530,-0.073739,35444
1,binding_gc_start,-0.065308,0.019259,35444
2,binding_gc_mid,-0.112312,-0.142181,35444
3,binding_gc_end,-0.072374,-0.026175,35444


In [3]:
enriched = add_utr_gc(enriched)
spearman_vs_inhibition(enriched, ["utr5_gc", "utr3_gc"])

,feature,spearman_overall,spearman_by_gene,n
0,utr5_gc,-0.108110,NaN,35444
1,utr3_gc,-0.178788,NaN,34398


In [4]:
enriched = add_binding_features(enriched, with_mfe=True)
spearman_vs_inhibition(enriched, ["binding_mfe"])

  folded 500/9100 unique slices
  folded 1000/9100 unique slices
  folded 1500/9100 unique slices
  folded 2000/9100 unique slices
  folded 2500/9100 unique slices
  folded 3000/9100 unique slices
  folded 3500/9100 unique slices
  folded 4000/9100 unique slices
  folded 4500/9100 unique slices
  folded 5000/9100 unique slices
  folded 5500/9100 unique slices
  folded 6000/9100 unique slices
  folded 6500/9100 unique slices
  folded 7000/9100 unique slices
  folded 7500/9100 unique slices
  folded 8000/9100 unique slices
  folded 8500/9100 unique slices
  folded 9000/9100 unique slices


,feature,spearman_overall,spearman_by_gene,n
0,binding_mfe,0.111227,0.11807,35444
